# `test_pipeline` — training entry point

(see `docs/model_design.md` §8.6)

There is intentionally no CLI: training is a one-shot exploratory activity. This notebook is structured so the expensive stage (preprocessing) runs once per kernel session, while Tucker/DMD/OU hyperparameters can be iterated on cheaply against the cached result.

Two workflows:
- **Iterate on hyperparameters** (this notebook, cells below) — explore Tucker rank / DMD rank against a cached preprocessing pass, then assemble the chosen combination into a `WeatherModel` via `from_components`.
- **Just train the whole thing** (bottom of this notebook) — once hyperparameters are settled, a single `WeatherModel(...).fit(...)` call for production runs.

## Cell 1 — Setup

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn.pipeline import Pipeline

from core.config import SNAPSHOT_CADENCE_SECONDS

# `./data` is a symlink to the real archive, laid out as
# <data_dir>/<YYYY>/<MM>/<DD>/<HH>/<variable>.npy (see docs/model_design.md §2.1).
DATA_DIR = Path("../data")
CACHE_DIR = Path("../cache")

# Adjust to the archive's actual coverage. docs/model_design.md §6 recommends
# 2 years training + 1 year validation (held out, used in Cell 10).
T_START = datetime(2023, 1, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 1, 1, tzinfo=timezone.utc)
HELD_OUT_START = T_END
HELD_OUT_END = T_END + timedelta(days=365)

VARIABLES = ["sigwh", "wsh", "wsp", "wsd", "was", "wad"]

SEED = 42
rng = np.random.default_rng(SEED)

## Cell 2 — Ingest a training window

In [ ]:
from train.ingest import load_training_window

X_raw = load_training_window(DATA_DIR, T_START, T_END, VARIABLES)
X_raw

## Cell 3 — Build and fit the preprocessing Pipeline

`memory=CACHE_DIR` (see docs/model_design.md §8.6, §9 gotcha) means restarting
the kernel and re-running cells 1–3 skips this expensive stage on the second
pass, as long as the inputs and step parameters are unchanged. Turn caching
off (`memory=None`) for production runs — it wastes disk.

In [ ]:
from train.climatology import ClimatologyRemover
from train.copula import GaussianCopula
from train.directional import DirectionalDecomposer
from train.reproject import Reprojector
from train.standardize import PerCellStandardizer

preprocess = Pipeline(
    [
        ("directional", DirectionalDecomposer()),
        ("reproject", Reprojector(target_grid="wave")),
        ("climatology", ClimatologyRemover(diurnal_variables=["sst"])),
        ("standardize", PerCellStandardizer()),
        ("gaussianize", GaussianCopula(n_bins=500)),
    ],
    memory=str(CACHE_DIR),
)
Z = preprocess.fit_transform(X_raw)  # slow: dominated by the copula LUT fit
Z

## Cell 4 — Inspect: `Z` at random cells should be standard normal

In [ ]:
variable_names = tuple(Z.data_vars)
n_lat, n_lon = Z[variable_names[0]].shape[1:]

fig, axes = plt.subplots(1, len(variable_names), figsize=(4 * len(variable_names), 4))
for ax, name in zip(axes, variable_names):
    lat_idx, lon_idx = rng.integers(0, n_lat), rng.integers(0, n_lon)
    series = Z[name].values[:, lat_idx, lon_idx]
    series = series[~np.isnan(series)]
    if series.size > 0:
        stats.probplot(series, dist="norm", plot=ax)
    ax.set_title(name)
fig.tight_layout()

## Cell 5 — Mask and stack into `(T, N_ocean, V)`

A cell is "ocean" if every modeled variable is non-NaN at every training
snapshot (see docs/model_design.md §5.1). This is the same mask `WeatherModel.fit`
computes internally; done explicitly here since Tucker needs a plain array,
not an `xr.Dataset`.

In [ ]:
ocean_mask = np.ones((n_lat, n_lon), dtype=bool)
for name in variable_names:
    ocean_mask &= ~np.isnan(Z[name].values).any(axis=0)

n_times = Z[variable_names[0]].shape[0]
n_ocean = int(ocean_mask.sum())
Z_array = np.empty((n_times, n_ocean, len(variable_names)), dtype=np.float64)
for i, name in enumerate(variable_names):
    Z_array[:, :, i] = Z[name].values[:, ocean_mask]

print(f"{n_ocean} ocean cells out of {n_lat * n_lon}")
Z_array.shape

## Cell 6 — Try Tucker at different ranks

In [ ]:
from train.tucker import hosvd, tucker_inverse

tucker_ranks = [200, 400]
tuckers = {r: hosvd(Z_array, r_ocean=r, r_var=6, rng=rng) for r in tucker_ranks}

reconstruction_error = {}
for r, tucker in tuckers.items():
    from train.tucker import tucker_transform

    core = tucker_transform(Z_array, tucker)
    reconstructed = tucker_inverse(core, tucker)
    reconstruction_error[r] = float(np.sqrt(np.mean((Z_array - reconstructed) ** 2)))

plt.plot(list(reconstruction_error), list(reconstruction_error.values()), "o-")
plt.xlabel("r_ocean")
plt.ylabel("RMS reconstruction error")

# Pick one.
R_OCEAN = 400
R_VAR = 6
tucker = tuckers[R_OCEAN]
core = tucker_transform(Z_array, tucker)

## Cell 7 — Try DMD at different `r_dmd` against the chosen Tucker

In [ ]:
from train.dmd import exact_dmd

dmd_ranks = [50, 100]
dmds = {
    r: exact_dmd(core, r_dmd=r, dt=SNAPSHOT_CADENCE_SECONDS, rng=rng) for r in dmd_ranks
}

fig, ax = plt.subplots()
for r, dmd in dmds.items():
    ax.scatter(dmd.Lambda.real, dmd.Lambda.imag, label=f"r_dmd={r}", alpha=0.6)
theta = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), "k--", linewidth=1)
ax.set_xlabel("Re(λ)")
ax.set_ylabel("Im(λ)")
ax.legend()

# Pick one.
R_DMD = 100
dmd = dmds[R_DMD]

## Cell 8 — OU fit against the chosen DMD

In [ ]:
from train.dmd import dmd_residuals
from train.ou import fit_ou

residuals = dmd_residuals(core, dmd)
ou = fit_ou(residuals, dt=SNAPSHOT_CADENCE_SECONDS)
ou.theta, ou.sigma

## Cell 9 — Assemble into a `WeatherModel` and save

In [ ]:
import sklearn

from weather.model import ModelMetadata, WeatherModel

metadata = ModelMetadata(
    trained_at=datetime.now(timezone.utc),
    t_train_start=T_START,
    t_train_end=T_END,
    variables=variable_names,
    r_ocean=R_OCEAN,
    r_var=R_VAR,
    r_dmd=R_DMD,
    git_commit="unknown",
    sklearn_version=sklearn.__version__,
    numpy_version=np.__version__,
)
model = WeatherModel.from_components(
    preprocess=preprocess,
    tucker=tucker,
    dmd=dmd,
    ou=ou,
    ocean_mask=ocean_mask,
    metadata=metadata,
)
model.save(Path("../models/v0.1.0"))

## Cell 10 — Validation smoke test

In [ ]:
from weather import full_report, sample_batch

lats = rng.uniform(-75.0, -60.0, size=10)
lons = rng.uniform(0.0, 360.0, size=10)
times = np.full(10, np.datetime64(T_START.replace(tzinfo=None), "s"))
samples = sample_batch(model, lats, lons, times, ["sigwh"], seed=42)
print(samples)

full_report(
    model,
    DATA_DIR,
    HELD_OUT_START,
    HELD_OUT_END,
    output=Path("../reports/v0.1.0_validation.pdf"),
    seed=42,
)

---
## "Just train the whole thing" (production runs)

Once hyperparameters have settled from the exploration above, a single call
does the whole pipeline — no manual mask/stack/from_components needed:

In [ ]:
production_model = WeatherModel(
    t_start=T_START,
    t_end=T_END,
    variables=VARIABLES,
    r_ocean=R_OCEAN,
    r_var=R_VAR,
    r_dmd=R_DMD,
    random_state=SEED,
)
production_model.fit(DATA_DIR)
production_model.save(Path("../models/v0.1.0"))